# Neural Network with PyTorch

## Binary Classification for Breast Cancer Diagnosis

Dataset: Breast Cancer Wisconsin (Diagnostic)
https://archive.ics.uci.edu/dataset/17/breast+cancer+wisconsin+diagnostic

This version uses PyTorch through `skorch`, which exposes the network through the scikit-learn interface.

In [1]:
import pandas as pd
import numpy as np
import torch
import torch.nn as nn
import sklearn
import skorch
from sklearn.model_selection import train_test_split, cross_val_score, GridSearchCV
from skorch import NeuralNetBinaryClassifier
from torchinfo import summary as torch_summary


In [2]:
pd.__version__, torch.__version__, sklearn.__version__, skorch.__version__


('3.0.6', '2.14.1+cpu', '1.8.0', '1.4.0')

### Load the Data

**Load the dataset and create the training and test sets with `train_test_split`, using 25% of the observations for testing.**

In [3]:
X = pd.read_csv('data/breast_features.csv')
y = pd.read_csv('data/breast_labels.csv')

X_treinamento, X_teste, y_treinamento, y_teste = train_test_split(
    X, y, test_size=0.25, random_state=0
)

In [4]:
X_treinamento.shape, y_treinamento.shape

((426, 30), (426, 1))

In [5]:
X_teste.shape, y_teste.shape

((143, 30), (143, 1))

### Neural Network Structure and Initial Test

In [6]:
from skorch import NeuralNetBinaryClassifier
from sklearn.model_selection import cross_val_score
import torch
import torch.nn as nn


**Create the neural network with the following configuration:**

**An input layer with 30 features.**

**A dense hidden layer with 16 neurons. Why was this size selected?**

Sixteen neurons provide an intermediate size between the 30 input features and the single output. This is not a fixed rule and should be compared with other configurations.

**Use ReLU activation in the hidden layer and initialize its weights with the random uniform initializer.**

**Add an output layer for binary classification. How are logits converted into probabilities?**

The classifier applies a sigmoid transformation to constrain probabilities to the interval from 0 to 1. The result is interpreted as the probability of the positive class.

**Why is a simple layer sequence appropriate for this network?**

The network is a straightforward stack of layers without branches.

In [7]:
class RedeNeuralModule(nn.Module):
    def __init__(self, n_features=30, neurons=16):
        super().__init__()
        self.oculta = nn.Linear(n_features, neurons)
        self.relu = nn.ReLU()
        self.saida = nn.Linear(neurons, 1)
        nn.init.uniform_(self.oculta.weight)
        nn.init.uniform_(self.saida.weight)

    def forward(self, X):
        x = self.relu(self.oculta(X))
        x = self.saida(x)  # logits (sem sigmoid); NeuralNetBinaryClassifier aplica BCEWithLogitsLoss
        return x.squeeze(-1)

rede_neural = NeuralNetBinaryClassifier(
    module=RedeNeuralModule,
    max_epochs=100,
    batch_size=10,
    verbose=0,
)


**Explain the values displayed by the network summary.**

- **Layer (type)**: name and type of each network layer.
- **Output Shape**: dimensions produced by each layer. `(None, 16)` means that every sample produces 16 values, while `None` allows any batch size.
- **Param #**: number of trainable parameters. The 16-neuron dense layer has (30 inputs × 16 neurons) + 16 biases = 496 parameters. The output layer has (16 × 1) + 1 bias = 17 parameters, for a total of 513.
- **Trainable params**: parameters updated during training.
- **Non-trainable params**: fixed parameters, such as parameters in frozen layers.

In [8]:
rede_neural.initialize()
torch_summary(rede_neural.module_, input_size=(1, 30))


Layer (type:depth-idx)                   Output Shape              Param #
RedeNeuralModule                         [1]                       --
├─Linear: 1-1                            [1, 16]                   496
├─ReLU: 1-2                              [1, 16]                   --
├─Linear: 1-3                            [1, 1]                    17
Total params: 513
Trainable params: 513
Non-trainable params: 0
Total mult-adds (Units.MEGABYTES): 0.00
Input size (MB): 0.00
Forward/backward pass size (MB): 0.00
Params size (MB): 0.00
Estimated Total Size (MB): 0.00

**Configure the Adam optimizer, binary cross-entropy loss, and binary accuracy evaluation.**

**What do optimizers do, and how does Adam work?**

Optimizers update network weights to minimize the loss function. Adam (Adaptive Moment Estimation) combines two ideas:
- **Momentum**: accumulates past gradients to smooth updates and reduce oscillation.
- **RMSProp**: adapts the learning rate for each parameter based on the magnitude of recent gradients.
Together, these mechanisms adapt the update applied to each parameter during training.

**Train the configured network with `.fit`. With `batch_size=10`, how many batches are processed in total?**

The training set contains 426 samples (75% of 569). With `batch_size=10`, the model processes ⌈426 / 10⌉ = 43 batches per epoch × 100 epochs = 4,300 weight updates.

**Train the model for 100 epochs.**

In [9]:
_ = rede_neural.set_params(optimizer=torch.optim.Adam, criterion=nn.BCEWithLogitsLoss)


In [10]:
X_tr = X_treinamento.values.astype(np.float32)
y_tr = y_treinamento.values.astype(np.float32).ravel()
_ = rede_neural.fit(X_tr, y_tr)


**Generate predictions for the test data. Why does the classifier return values between 0 and 1?**

The sigmoid transformation constrains each probability to the interval from 0 to 1, representing the positive class probability.

In [11]:
X_te = X_teste.values.astype(np.float32)
previsoes = rede_neural.predict_proba(X_te)[:, 1]


**Convert the probabilities into binary predictions with a threshold of 0.5.**

In [12]:
previsoes = previsoes > 0.5

In [13]:
previsoes

array([False, False, False, False,  True,  True,  True,  True,  True,
        True, False, False,  True, False, False, False,  True, False,
       False, False, False, False, False, False, False, False, False,
       False, False, False, False, False, False, False,  True, False,
       False, False, False, False, False,  True, False,  True, False,
       False,  True, False,  True, False,  True, False, False,  True,
       False,  True,  True,  True,  True, False, False, False,  True,
       False, False, False, False, False, False,  True, False, False,
        True, False, False, False,  True,  True, False,  True, False,
       False, False, False, False,  True, False,  True, False, False,
        True, False,  True, False,  True,  True, False, False, False,
        True,  True,  True, False, False, False,  True, False, False,
       False, False, False, False, False, False,  True,  True, False,
        True, False, False, False, False, False, False, False, False,
       False,  True,

**Interpret the network result.**

With a threshold of 0.5, each probability is converted into one of the two classes. The evaluation reports performance on the test set.

In [14]:
y_te = y_teste.values.astype(np.float32).ravel()
acuracia = rede_neural.score(X_te, y_te)
print(f'Binary accuracy on the test set: {acuracia:.4f}')


Binary accuracy on the test set: 0.6643


## Network Layers and Optimization

**Add another dense hidden layer with 16 neurons, use the ReLU activation function, and initialize the weights with a random uniform initializer. What is the network's total number of parameters now?**

With the second hidden layer:
- Layer 1: (30×16) + 16 = 496 parameters
- Layer 2: (16×16) + 16 = 272 parameters
- Output: (16×1) + 1 = 17 parameters
- **Total: 785 parameters**

In [15]:
class RedeNeuralModule(nn.Module):
    def __init__(self, n_features=30, neurons=16):
        super().__init__()
        self.oculta1 = nn.Linear(n_features, neurons)
        self.oculta2 = nn.Linear(neurons, neurons)
        self.relu = nn.ReLU()
        self.saida = nn.Linear(neurons, 1)
        for camada in (self.oculta1, self.oculta2, self.saida):
            nn.init.uniform_(camada.weight)

    def forward(self, X):
        x = self.relu(self.oculta1(X))
        x = self.relu(self.oculta2(x))
        x = self.saida(x)
        return x.squeeze(-1)

rede_neural = NeuralNetBinaryClassifier(
    module=RedeNeuralModule,
    max_epochs=100,
    batch_size=10,
    verbose=0,
)
_ = rede_neural.initialize()


In [16]:
torch_summary(rede_neural.module_, input_size=(1, 30))


Layer (type:depth-idx)                   Output Shape              Param #
RedeNeuralModule                         [1]                       --
├─Linear: 1-1                            [1, 16]                   496
├─ReLU: 1-2                              [1, 16]                   --
├─Linear: 1-3                            [1, 16]                   272
├─ReLU: 1-4                              [1, 16]                   --
├─Linear: 1-5                            [1, 1]                    17
Total params: 785
Trainable params: 785
Non-trainable params: 0
Total mult-adds (Units.MEGABYTES): 0.00
Input size (MB): 0.00
Forward/backward pass size (MB): 0.00
Params size (MB): 0.00
Estimated Total Size (MB): 0.00

**The optimizer parameters `learning_rate` and `clipvalue` are added below. What do they do?**

- **learning_rate (0.001)** controls the step size in gradient descent. Very high values can make the model diverge, while very low values slow down training. A value of 0.001 is Adam's default.
- **clipvalue (0.5)** limits gradient values during backpropagation. It helps prevent exploding gradients from destabilizing training.

In [17]:
from skorch.callbacks import Callback

class ClipGradByValue(Callback):
    def on_grad_computed(self, net, named_parameters, **kwargs):
        torch.nn.utils.clip_grad_value_(net.module_.parameters(), clip_value=0.5)

_ = rede_neural.set_params(
    optimizer=torch.optim.Adam,
    optimizer__lr=0.001,
    criterion=nn.BCEWithLogitsLoss,
    callbacks=[ClipGradByValue()],
)


**Test the neural network again. Did adding a layer improve or worsen the results?**

The second layer increases the model's capacity but does not guarantee better test performance. If training performance improves while test performance worsens, the network is probably overfitting.

In [18]:
_ = rede_neural.fit(X_tr, y_tr)


In [19]:
previsoes = rede_neural.predict_proba(X_te)[:, 1]


In [20]:
acuracia = rede_neural.score(X_te, y_te)
print(f'Binary accuracy on the test set: {acuracia:.4f}')


Binary accuracy on the test set: 0.7902


## K-Fold Cross Validation

**How is this network configured for cross-validation?**

In cross-validation, each fold is used once for validation while the remaining folds are used for training. With 10 folds, the network is trained 10 times. `NeuralNetBinaryClassifier` exposes the model through the scikit-learn API.

**Why calculate the standard deviation of the results?**

The standard deviation shows how much accuracy changes across folds. High values indicate greater sensitivity to the data split.

In [21]:
from skorch import NeuralNetBinaryClassifier
from skorch.callbacks import Callback
from sklearn.model_selection import cross_val_score
import torch.nn as nn
import torch


In [22]:
class ClipGradByValue(Callback):
    def on_grad_computed(self, net, named_parameters, **kwargs):
        torch.nn.utils.clip_grad_value_(net.module_.parameters(), clip_value=0.5)

def criar_rede():
    return NeuralNetBinaryClassifier(
        module=RedeNeuralModule,
        max_epochs=100,
        batch_size=10,
        optimizer=torch.optim.Adam,
        optimizer__lr=0.001,
        criterion=nn.BCEWithLogitsLoss,
        callbacks=[ClipGradByValue()],
        verbose=0,
    )


In [23]:
rede_neural = criar_rede()


In [24]:
X_np = X.values.astype(np.float32)
y_np = y.values.astype(np.float32).ravel()
resultados = cross_val_score(estimator=rede_neural, X=X_np, y=y_np, cv=10, scoring='accuracy')


In [25]:
resultados

array([0.71929825, 0.68421053, 0.71929825, 0.84210526, 0.87719298,
       0.84210526, 0.75438596, 0.84210526, 0.80701754, 0.71428571])

In [26]:
print(f'Mean accuracy:      {resultados.mean():.4f}')
print(f'Standard deviation: {resultados.std():.4f}')

Mean accuracy:      0.7802
Standard deviation: 0.0658


## Overfitting and Dropout

**Apply 20% dropout to both hidden layers to reduce overfitting. How do the results and standard deviation change?**

Dropout disables 20% of the activations during training. Compare configurations using the validation mean and standard deviation without assuming that dropout will necessarily improve the result.

In [27]:
class RedeNeuralDropoutModule(nn.Module):
    def __init__(self, n_features=30, neurons=16, dropout=0.2):
        super().__init__()
        self.oculta1 = nn.Linear(n_features, neurons)
        self.drop1 = nn.Dropout(dropout)
        self.oculta2 = nn.Linear(neurons, neurons)
        self.drop2 = nn.Dropout(dropout)
        self.relu = nn.ReLU()
        self.saida = nn.Linear(neurons, 1)
        for camada in (self.oculta1, self.oculta2, self.saida):
            nn.init.uniform_(camada.weight)

    def forward(self, X):
        x = self.drop1(self.relu(self.oculta1(X)))
        x = self.drop2(self.relu(self.oculta2(x)))
        x = self.saida(x)
        return x.squeeze(-1)

def criar_rede_dropout():
    return NeuralNetBinaryClassifier(
        module=RedeNeuralDropoutModule,
        max_epochs=100,
        batch_size=10,
        optimizer=torch.optim.Adam,
        optimizer__lr=0.001,
        criterion=nn.BCEWithLogitsLoss,
        callbacks=[ClipGradByValue()],
        verbose=0,
    )


In [28]:
rede_neural = criar_rede_dropout()


In [29]:
resultados = cross_val_score(estimator=rede_neural, X=X_np, y=y_np, cv=10, scoring='accuracy')


In [30]:
resultados

array([0.64912281, 0.75438596, 0.70175439, 0.85964912, 0.52631579,
       0.75438596, 0.8245614 , 0.57894737, 0.63157895, 0.71428571])

In [31]:
print(f'Mean accuracy with dropout:      {resultados.mean():.4f}')
print(f'Standard deviation with dropout: {resultados.std():.4f}')

Mean accuracy with dropout:      0.6995
Standard deviation with dropout: 0.0996


## Hyperparameter Tuning

Hyperparameter tuning compares configurations defined before training.
Examples of hyperparameters include:

- Learning rate
- Number of trees in a random forest
- Number of layers or neurons in a neural network
- Kernel type in an SVM

Common search strategies include:

- Grid Search
- Random Search
- Bayesian Optimization, etc.

**How was the neural network configured for tuning?**

The network class exposes the parameters to be compared. `NeuralNetBinaryClassifier` adapts the model to the scikit-learn interface, and `GridSearchCV` evaluates the combinations with five folds.

In [32]:
class RedeNeuralTunavelModule(nn.Module):
    def __init__(self, n_features=30, neurons=16, activation=nn.ReLU):
        super().__init__()
        self.oculta1 = nn.Linear(n_features, neurons)
        self.drop1 = nn.Dropout(0.2)
        self.oculta2 = nn.Linear(neurons, neurons)
        self.drop2 = nn.Dropout(0.2)
        self.ativacao = activation()
        self.saida = nn.Linear(neurons, 1)

    def forward(self, X):
        x = self.drop1(self.ativacao(self.oculta1(X)))
        x = self.drop2(self.ativacao(self.oculta2(x)))
        x = self.saida(x)
        return x.squeeze(-1)

rede_neural = NeuralNetBinaryClassifier(
    module=RedeNeuralTunavelModule,
    criterion=nn.BCEWithLogitsLoss,
    verbose=0,
)


In [33]:
# rede_neural was created in the previous cell


In [34]:
parametros = {
    'batch_size': [10, 30],
    'max_epochs': [50, 100],
    'optimizer': [torch.optim.Adam, torch.optim.SGD],
    'module__activation': [nn.ReLU, nn.Tanh],
    'module__neurons': [16, 8],
}


In [35]:
# Use this reduced search space if the previous configuration is too expensive
parametros = {
    'batch_size': [10, 30],
    'max_epochs': [50],
    'optimizer': [torch.optim.Adam],
    'module__activation': [nn.ReLU],
    'module__neurons': [16],
}


In [36]:
parametros

{'batch_size': [10, 30],
 'max_epochs': [50],
 'optimizer': [torch.optim.adam.Adam],
 'module__activation': [torch.nn.modules.activation.ReLU],
 'module__neurons': [16]}

In [37]:
grid_search = GridSearchCV(estimator=rede_neural, param_grid=parametros,
                           scoring='accuracy', cv=5)


In [38]:
grid_search = grid_search.fit(X_np, y_np)


In [39]:
print(grid_search)

GridSearchCV(cv=5,
             estimator=NeuralNetBinaryClassifier(_params_to_validate=set(), batch_size=128, callbacks=None, compile=False, dataset=<class 'skorch.dataset.Dataset'>, device='cpu', iterator_train=<class 'torch.utils.data.dataloader.DataLoader'>, iterator_valid=<class 'torch.utils.data.dataloader.DataLoader'>, lr=0.01, max_epochs=10, module=<class '__main__.Re...Module'>, optimizer=<class 'torch.optim.sgd.SGD'>, predict_nonlinearity='auto', torch_load_kwargs=None, use_caching='auto', verbose=0, warm_start=False),
             param_grid={'batch_size': [10, 30], 'max_epochs': [50],
                         'module__activation': [<class 'torch.nn.modules.activation.ReLU'>],
                         'module__neurons': [16],
                         'optimizer': [<class 'torch.optim.adam.Adam'>]},
             scoring='accuracy')


In [40]:
melhores_parametros = grid_search.best_params_
print(melhores_parametros)

{'batch_size': 30, 'max_epochs': 50, 'module__activation': <class 'torch.nn.modules.activation.ReLU'>, 'module__neurons': 16, 'optimizer': <class 'torch.optim.adam.Adam'>}


In [41]:
melhor_precisao = grid_search.best_score_
print(melhor_precisao)

0.7980593075609377


**Can the neural network's accuracy be improved further?**

Possible strategies include:
- **Expand the grid search space** by testing more combinations of neurons, layers, optimizers, and learning rates.
- **Scale the data** with `StandardScaler` or `MinMaxScaler` to improve convergence.
- **Train for more epochs** while monitoring validation performance to control overfitting.
- **Add layers** when the problem requires a model with greater capacity.
- **Use batch normalization** to stabilize activations and training.
- **Use early stopping** when validation loss stops improving.

# Saving the Neural Network

In [42]:
# Train the final model with the selected parameters before saving it
classificador = NeuralNetBinaryClassifier(
    module=RedeNeuralDropoutModule,
    max_epochs=100,
    batch_size=10,
    optimizer=torch.optim.Adam,
    optimizer__lr=0.001,
    criterion=nn.BCEWithLogitsLoss,
    callbacks=[ClipGradByValue()],
    verbose=0,
)
_ = classificador.fit(X_tr, y_tr)


In [43]:
from pathlib import Path

model_path = Path('artifacts') / 'breast_cancer_classifier.pt'
model_path.parent.mkdir(parents=True, exist_ok=True)
print(f'Saving model to: {model_path}')
classificador.save_params(f_params=model_path)


Saving model to: artifacts\breast_cancer_classifier.pt


In [44]:
classificador_novo = NeuralNetBinaryClassifier(
    module=RedeNeuralDropoutModule,
    criterion=nn.BCEWithLogitsLoss,
)
classificador_novo.initialize()
classificador_novo.load_params(f_params=model_path)


In [45]:
torch_summary(classificador_novo.module_, input_size=(1, 30))


Layer (type:depth-idx)                   Output Shape              Param #
RedeNeuralDropoutModule                  [1]                       --
├─Linear: 1-1                            [1, 16]                   496
├─ReLU: 1-2                              [1, 16]                   --
├─Dropout: 1-3                           [1, 16]                   --
├─Linear: 1-4                            [1, 16]                   272
├─ReLU: 1-5                              [1, 16]                   --
├─Dropout: 1-6                           [1, 16]                   --
├─Linear: 1-7                            [1, 1]                    17
Total params: 785
Trainable params: 785
Non-trainable params: 0
Total mult-adds (Units.MEGABYTES): 0.00
Input size (MB): 0.00
Forward/backward pass size (MB): 0.00
Params size (MB): 0.00
Estimated Total Size (MB): 0.00